# Grid-Ablation Trials

Isolates the effect of each chorus-boundary improvement, one change per trial,
scored in the **time domain** (grid-independent) on a **frozen test set**.

| Trial | Change | Isolates |
|-------|--------|----------|
| T0 | pretrained model, librosa grid, smoothing | reference baseline |
| T1 | grid alignment audit (no model) | H1: are beat_this downbeats closer to ground-truth boundaries? |
| T2a | retrain on librosa grid (frozen split) | honest baseline |
| T2b | retrain on beat_this grid (frozen split) | H2: does the grid help? |
| T3 | Viterbi decoding (both models) | H3: does decoding help, independent of grid? |
| T4 | downbeat snapping (both models) | H4: does snapping fix residual sub-bar offset? |

**Prerequisites** (run once on the pod before this notebook — see `RUNPOD_README.md`):
- `python scripts/freeze_split.py`
- `python scripts/preprocess.py --segments-dir data/seg_librosa --labels-dir data/lab_librosa`
- `python scripts/preprocess.py --grid-source beat_this --device cuda --segments-dir data/seg_beatthis --labels-dir data/lab_beatthis`

## Setup

In [ ]:
import os, sys, warnings
import numpy as np, pandas as pd, torch, librosa
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

REPO = os.path.abspath("..")            # this notebook lives in notebooks/
sys.path.insert(0, REPO)

from pytorch_core.audio_processor import process_audio
from pytorch_core.model import smooth_predictions, load_CRNN_model
from pytorch_core.decoding import viterbi_chorus
from pytorch_core.downbeats import track_downbeats, snap_chorus_segments, is_available
from pytorch_core import evaluation as ev
from scripts.inference import load_config, load_model

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SMOKE = False                            # True -> 5 test songs, 2 training epochs

CONFIG = load_config(os.path.join(REPO, "config", "default.yaml"))
AUDIO_DIR = os.path.join(REPO, "data", "audio", "processed")
LABELS = pd.read_csv(os.path.join(REPO, "data", "clean_labeled.csv"))
RESULTS_CSV = os.path.join(REPO, "results", "trials.csv")
os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)

def read_ids(name):
    with open(os.path.join(REPO, "data", "splits", name)) as f:
        return [ln.strip() for ln in f if ln.strip()]

TRAIN_IDS, VAL_IDS, TEST_IDS = read_ids("train_songs.txt"), read_ids("val_songs.txt"), read_ids("test_songs.txt")
if SMOKE:
    TEST_IDS = TEST_IDS[:5]
print(f"device={DEVICE}  train={len(TRAIN_IDS)} val={len(VAL_IDS)} test={len(TEST_IDS)}  beat_this={is_available()}")

## Shared helpers

In [ ]:
def song_rows(song_id):
    return LABELS[LABELS["SongID"].astype(str) == str(song_id)]

def true_segments(song_id):
    rows = song_rows(song_id)
    rows = rows[rows["label"] == "chorus"].sort_values("start_time")
    starts, ends = [], []
    for _, r in rows.iterrows():
        if ends and abs(r["start_time"] - ends[-1]) < 1e-6:
            ends[-1] = float(r["end_time"])
        else:
            starts.append(float(r["start_time"])); ends.append(float(r["end_time"]))
    return starts, ends

def song_meta(song_id):
    r = song_rows(song_id).iloc[0]
    bpm = r.get("sp_tempo", np.nan)
    bpm = None if pd.isna(bpm) or bpm == 0 else float(bpm)
    ts = r.get("sp_time_signature", np.nan)
    ts = 4 if pd.isna(ts) or ts == 0 else int(ts)
    return bpm, ts

def audio_path(song_id):
    return os.path.join(AUDIO_DIR, f"{song_id}.mp3")

def segments_from_binary(binary, grid_times):
    starts, ends = [], []
    idx = np.where(np.asarray(binary) == 1)[0]
    if idx.size == 0:
        return starts, ends
    grp = [idx[0]]
    for i in idx[1:]:
        if i == grp[-1] + 1:
            grp.append(i)
        else:
            starts.append(float(grid_times[grp[0]])); ends.append(float(grid_times[grp[-1] + 1])); grp = [i]
    starts.append(float(grid_times[grp[0]])); ends.append(float(grid_times[grp[-1] + 1]))
    return starts, ends

In [ ]:
def predict(model, song_id, grid_source, decode="smooth", snap=False):
    # Returns (pred_starts, pred_ends, beat_period, duration), or None if unavailable.
    path = audio_path(song_id)
    if not os.path.exists(path):
        return None
    bpm, ts = song_meta(song_id)
    # Audio is already silence-stripped and labels were annotated on it (no re-trim).
    processed, af = process_audio(path, trim_silence=False,
                                  sr=CONFIG["data"]["sr"], hop_length=CONFIG["data"]["hop_length"],
                                  bpm=bpm, time_signature=ts, grid_source=grid_source, device=DEVICE)
    if processed is None:
        return None
    with torch.no_grad():
        out = model(torch.tensor(processed, dtype=torch.float32).to(DEVICE)).cpu().numpy().squeeze()
    n_meters = min(len(af.meter_grid) - 1, len(out))
    probs = out[:n_meters]
    binary = viterbi_chorus(probs, switch_penalty=2.0, min_bars=4) if decode == "viterbi" \
        else smooth_predictions(probs)
    grid_times = librosa.frames_to_time(af.meter_grid, sr=af.sr, hop_length=af.hop_length)
    starts, ends = segments_from_binary(binary, grid_times)
    beat_period = 60.0 / af.tempo if af.tempo else None
    if snap and starts and is_available():
        _, downbeats = track_downbeats(path, device=DEVICE)
        if len(downbeats) >= 2:
            rms = np.asarray(af.rms).ravel()
            rms_t = librosa.frames_to_time(np.arange(rms.size), sr=af.sr, hop_length=af.hop_length)
            starts, ends = snap_chorus_segments(starts, ends, downbeats, energy=rms, energy_times=rms_t)
    duration = len(af.y) / af.sr
    return starts, ends, beat_period, duration


def run_trial(trial_id, description, model, grid_source, decode="smooth", snap=False, test_ids=None):
    rows = []
    for sid in (test_ids or TEST_IDS):
        r = predict(model, sid, grid_source, decode=decode, snap=snap)
        if r is None:
            continue
        ps, pe, bp, dur = r
        ts_, te_ = true_segments(sid)
        rows.append(ev.score_song(ps, pe, ts_, te_, duration=dur, beat_period=bp))
    agg = ev.aggregate(rows)
    record = {"trial": trial_id, "description": description, "grid": grid_source,
              "decode": decode, "snap": int(snap), "n_songs": len(rows), **agg}
    pd.DataFrame([record]).to_csv(RESULTS_CSV, mode="a",
                                  header=not os.path.exists(RESULTS_CSV), index=False)
    print(f"[{trial_id}] {description}  (n={len(rows)})")
    for k in ("f1_mean", "median_abs_err_s_median", "median_abs_err_beats_median",
              "hit_rate_70ms_mean", "hit_rate_500ms_mean"):
        if k in agg:
            print(f"    {k}: {agg[k]:.3f}")
    return record

## T0 — reference baseline

The released `crnn_v1.pt` on the librosa grid with standard smoothing.
**Reference only:** it was trained on the original `os.listdir`-order split, which
may overlap this frozen test set. The contamination-free baseline is **T2a**.

In [ ]:
pretrained = load_CRNN_model(os.path.join(REPO, "models", "CRNN_pytorch", "crnn_v1.pt"))
pretrained.to(DEVICE).eval()
run_trial("T0", "pretrained, librosa grid, smooth (reference)", pretrained, "librosa");

## T1 — grid alignment audit (no model)

For every ground-truth chorus boundary, distance to the nearest **librosa grid
line** vs the nearest **beat_this downbeat**, expressed in beats. Both are bar
grids, so this is apples-to-apples. A concentration of librosa errors near **1
beat** with beat_this near 0 is the phase-error / delayed-drop failure, and is
the evidence for H1.

In [ ]:
def grid_line_times(song_id, grid_source):
    path = audio_path(song_id)
    bpm, ts = song_meta(song_id)
    _, af = process_audio(path, trim_silence=False, sr=CONFIG["data"]["sr"],
                          hop_length=CONFIG["data"]["hop_length"], bpm=bpm, time_signature=ts,
                          grid_source=grid_source, device=DEVICE)
    beat_period = 60.0 / af.tempo if af.tempo else None
    return librosa.frames_to_time(af.meter_grid, sr=af.sr, hop_length=af.hop_length), beat_period

librosa_err, beatthis_err = [], []
for sid in TEST_IDS:
    if not os.path.exists(audio_path(sid)):
        continue
    ts_, te_ = true_segments(sid)
    truth = np.array(ts_ + te_)
    if truth.size == 0:
        continue
    lt, bp = grid_line_times(sid, "librosa")
    librosa_err += list(np.min(np.abs(lt[None, :] - truth[:, None]), axis=1) / bp)
    if is_available():
        _, downbeats = track_downbeats(audio_path(sid), device=DEVICE)
        if len(downbeats):
            db = np.asarray(downbeats)
            beatthis_err += list(np.min(np.abs(db[None, :] - truth[:, None]), axis=1) / bp)

def summarize(errs, name):
    if not errs:
        print(f"{name}: no data"); return
    e = np.abs(np.array(errs))
    print(f"{name}: median |err| = {np.median(e):.3f} beats, within 0.25 beat = {np.mean(e <= 0.25):.1%}")

summarize(librosa_err, "librosa grid")
summarize(beatthis_err, "beat_this downbeats")

plt.figure(figsize=(9, 4))
plt.hist(librosa_err, bins=40, range=(0, 2), alpha=0.6, label="librosa grid")
if beatthis_err:
    plt.hist(beatthis_err, bins=40, range=(0, 2), alpha=0.6, label="beat_this downbeats")
plt.xlabel("distance from ground-truth boundary (beats)"); plt.ylabel("count")
plt.title("T1 — grid alignment to ground-truth boundaries"); plt.legend(); plt.show()

## T2 — retrain on each grid (frozen split)

`train_model` builds datasets from the frozen train/val IDs (only songs whose
pickles exist in the given directory) and trains a fresh CRNN. T2a and T2b differ
only in which preprocessing directory they read, so their comparison isolates the
grid.

In [ ]:
from pytorch_core.data.dataset import ChorusDataset
from pytorch_core.training.trainer import Trainer
from pytorch_core.models.crnn import CRNN
from torch.utils.data import DataLoader

def available_ids(ids, seg_dir):
    return [s for s in ids if os.path.exists(os.path.join(seg_dir, f"{s}_data.pkl"))]

def train_model(seg_dir, lab_dir, ckpt_dir):
    epochs = 2 if SMOKE else CONFIG["training"]["epochs"]
    train_ds = ChorusDataset(available_ids(TRAIN_IDS, seg_dir), seg_dir, lab_dir, CONFIG)
    val_ds = ChorusDataset(available_ids(VAL_IDS, seg_dir), seg_dir, lab_dir, CONFIG)
    bs = CONFIG["training"]["batch_size"]
    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=bs, shuffle=False)
    torch.manual_seed(42); np.random.seed(42)
    trainer = Trainer(CRNN(CONFIG), CONFIG, train_loader, val_loader,
                      checkpoint_dir=ckpt_dir, device=DEVICE)
    trainer.train(epochs=epochs)
    return load_model(os.path.join(ckpt_dir, "best_model.pt"), CONFIG).to(DEVICE).eval()

In [ ]:
# T2a — honest baseline: retrain on the librosa grid under the frozen split.
model_librosa = train_model(os.path.join(REPO, "data", "seg_librosa"),
                            os.path.join(REPO, "data", "lab_librosa"),
                            os.path.join(REPO, "models", "ablation_librosa"))
run_trial("T2a", "retrained librosa grid, smooth", model_librosa, "librosa");

In [ ]:
# T2b — retrain on the beat_this grid under the same frozen split.
model_beatthis = train_model(os.path.join(REPO, "data", "seg_beatthis"),
                             os.path.join(REPO, "data", "lab_beatthis"),
                             os.path.join(REPO, "models", "ablation_beatthis"))
run_trial("T2b", "retrained beat_this grid, smooth", model_beatthis, "beat_this");

## T3 — Viterbi decoding (both models)

Replaces 0.5-threshold smoothing with the two-state Viterbi decoder + duration
prior. Running it on both models separates the decoding gain from the grid gain.

In [ ]:
run_trial("T3a", "librosa grid, viterbi", model_librosa, "librosa", decode="viterbi")
run_trial("T3b", "beat_this grid, viterbi", model_beatthis, "beat_this", decode="viterbi");

## T4 — downbeat snapping (both models)

Energy-aware snapping to beat_this downbeats on top of the Viterbi decode,
targeting residual sub-bar offset (the delayed-drop case).

In [ ]:
run_trial("T4a", "librosa grid, viterbi + snap", model_librosa, "librosa", decode="viterbi", snap=True)
run_trial("T4b", "beat_this grid, viterbi + snap", model_beatthis, "beat_this", decode="viterbi", snap=True);

## Summary

All trials, most-comparable metrics. Read T2b vs T2a for the grid effect, T3 vs
T2 for decoding, T4 vs T3 for snapping. Lower `median_abs_err_*` and higher
`f1_mean` / `hit_rate_*` are better.

In [ ]:
res = pd.read_csv(RESULTS_CSV)
cols = [c for c in ["trial", "description", "grid", "decode", "snap", "n_songs",
                    "f1_mean", "median_abs_err_s_median", "median_abs_err_beats_median",
                    "hit_rate_70ms_mean", "hit_rate_500ms_mean"] if c in res.columns]
res[cols].round(3)